# Module 03-01 — K-Means Clustering

**Learning objective:** Implement Lloyd's K-Means algorithm and K-Means++ initialization from scratch, evaluate cluster quality with the elbow method and silhouette score, and understand the algorithm's convergence properties and failure modes.

**Prerequisites:** 01-01 (NumPy & Tensor Speed), 01-09 (Calculus & Optimization Foundations)

---

## Part 0 — Setup & Prerequisites

K-Means is the most widely used clustering algorithm. Given $n$ data points and a target $k$, it partitions the data into $k$ clusters by minimising the **within-cluster sum of squares (WCSS)**:

$$\mathcal{L} = \sum_{j=1}^{k} \sum_{\mathbf{x}_i \in C_j} \|\mathbf{x}_i - \boldsymbol{\mu}_j\|^2$$

where $\boldsymbol{\mu}_j$ is the centroid of cluster $C_j$.

We build up the algorithm step by step: distance computation → random initialisation → Lloyd's alternating assignment/update → K-Means++ initialisation → cluster quality metrics → mini-batch variant → limitations.

In [ ]:
import sys
import time
import random
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.spatial.distance import cdist        # scipy allowed in Module 03

from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.metrics import silhouette_score as sklearn_silhouette_score

import torch
warnings.filterwarnings("ignore")

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy:  {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Configuration
N_SAMPLES   = 600    # number of data points for main dataset
N_CENTERS   = 4      # true number of clusters
N_FEATURES  = 2      # 2-D for easy visualisation
K_MAX       = 10     # maximum k to test in elbow method
MAX_ITER    = 300    # maximum Lloyd iterations
TOL         = 1e-4   # convergence tolerance (centroid shift)
N_INIT      = 10     # number of restarts for final model
BATCH_SIZE  = 64     # mini-batch size

In [ ]:
# Generate dataset
X_blobs, y_true = make_blobs(
    n_samples=N_SAMPLES,
    centers=N_CENTERS,
    cluster_std=0.8,
    random_state=SEED,
)

print(f"Dataset shape: {X_blobs.shape}")
print(f"True cluster labels: {np.unique(y_true)}")

# Visualise raw data (labels hidden — unsupervised setting)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], s=15, alpha=0.6, c="#888888")
axes[0].set_title("Raw data (labels hidden — unsupervised setting)")
axes[0].set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], s=15, alpha=0.7,
                c=y_true, cmap="tab10")
axes[1].set_title("Ground truth clusters (for reference only)")
axes[1].set_xlabel("Feature 1")
axes[1].set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

---

## Part 1 — Lloyd's K-Means from Scratch

### The Algorithm

Lloyd's algorithm alternates between two steps until convergence:

**Assignment step:** Assign each point $\mathbf{x}_i$ to the nearest centroid:
$$z_i = \arg\min_j \|\mathbf{x}_i - \boldsymbol{\mu}_j\|^2$$

**Update step:** Recompute each centroid as the mean of its assigned points:
$$\boldsymbol{\mu}_j = \frac{1}{|C_j|} \sum_{\mathbf{x}_i \in C_j} \mathbf{x}_i$$

The algorithm is guaranteed to converge (WCSS is non-increasing), but may converge to a local minimum depending on initialisation.

### Complexity

- Per iteration: $O(n \cdot k \cdot d)$ — compute distances from $n$ points to $k$ centroids in $d$ dimensions.
- Typically converges in 10–100 iterations.

> **See Module 1-10** for Big-O complexity analysis and vectorisation patterns.

In [ ]:
def pairwise_sq_distances(X: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """Compute squared Euclidean distances between each point and each centroid.

    Uses the identity ||a - b||^2 = ||a||^2 - 2a.b + ||b||^2 for efficiency,
    avoiding an explicit loop over centroids.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        centroids: Centroid matrix of shape (k, n_features).

    Returns:
        Distance matrix of shape (n_samples, k).
    """
    # ||x_i||^2 — shape (n, 1)
    X_sq = np.sum(X ** 2, axis=1, keepdims=True)
    # ||mu_j||^2 — shape (1, k)
    C_sq = np.sum(centroids ** 2, axis=1, keepdims=True).T
    # cross term — shape (n, k)
    cross = X @ centroids.T
    return X_sq - 2 * cross + C_sq


def assign_clusters(X: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """Assign each point to the nearest centroid (Lloyd assignment step).

    Args:
        X: Data matrix of shape (n_samples, n_features).
        centroids: Centroid matrix of shape (k, n_features).

    Returns:
        Integer label array of shape (n_samples,), values in 0..k-1.
    """
    dists = pairwise_sq_distances(X, centroids)
    return np.argmin(dists, axis=1)


def update_centroids(
    X: np.ndarray,
    labels: np.ndarray,
    k: int,
    old_centroids: np.ndarray,
) -> np.ndarray:
    """Recompute centroids as the mean of assigned points (Lloyd update step).

    If a cluster becomes empty, the centroid is kept at its previous position.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        labels: Current cluster assignments, shape (n_samples,).
        k: Number of clusters.
        old_centroids: Previous centroid positions, shape (k, n_features).

    Returns:
        Updated centroid matrix of shape (k, n_features).
    """
    new_centroids = np.empty_like(old_centroids)
    for j in range(k):
        mask = labels == j
        if mask.sum() == 0:
            # Empty cluster: keep old centroid (avoid NaN)
            new_centroids[j] = old_centroids[j]
        else:
            new_centroids[j] = X[mask].mean(axis=0)
    return new_centroids


def compute_wcss(X: np.ndarray, labels: np.ndarray, centroids: np.ndarray) -> float:
    """Compute within-cluster sum of squares (WCSS / inertia).

    Args:
        X: Data matrix of shape (n_samples, n_features).
        labels: Cluster assignments, shape (n_samples,).
        centroids: Centroid matrix of shape (k, n_features).

    Returns:
        Total WCSS (sum over all clusters of squared distances to centroid).
    """
    total = 0.0
    for j in range(centroids.shape[0]):
        mask = labels == j
        if mask.sum() > 0:
            diff = X[mask] - centroids[j]
            total += float(np.sum(diff ** 2))
    return total

### Random Initialisation

The simplest initialisation: pick $k$ data points uniformly at random as starting centroids. This is fast but can lead to poor local minima when clusters are poorly separated or very different in size.

In [ ]:
def init_random(X: np.ndarray, k: int, rng: np.random.RandomState) -> np.ndarray:
    """Initialise k centroids by sampling k points uniformly at random.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k: Number of clusters.
        rng: NumPy RandomState for reproducibility.

    Returns:
        Initial centroid matrix of shape (k, n_features).
    """
    indices = rng.choice(len(X), size=k, replace=False)
    return X[indices].copy()

In [ ]:
def lloyd_kmeans(
    X: np.ndarray,
    k: int,
    init: str = "random",
    max_iter: int = MAX_ITER,
    tol: float = TOL,
    seed: int = SEED,
) -> tuple:
    """Run Lloyd's K-Means algorithm with random or K-Means++ initialisation.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k: Number of clusters.
        init: Initialisation strategy — "random" or "kmeans++".
        max_iter: Maximum number of Lloyd iterations.
        tol: Convergence tolerance: stop when max centroid shift < tol.
        seed: Random seed for reproducibility.

    Returns:
        Tuple of (labels, centroids, wcss_history, n_iters) where:
            labels: Final cluster assignments, shape (n_samples,).
            centroids: Final centroid positions, shape (k, n_features).
            wcss_history: WCSS value after each iteration.
            n_iters: Number of iterations completed.
    """
    rng = np.random.RandomState(seed)

    if init == "random":
        centroids = init_random(X, k, rng)
    elif init == "kmeans++":
        centroids = init_kmeans_plus_plus(X, k, rng)
    else:
        raise ValueError(f"Unknown init strategy: {init!r}")

    wcss_history = []
    labels = np.zeros(len(X), dtype=int)

    for iteration in range(max_iter):
        # Assignment step
        labels = assign_clusters(X, centroids)
        wcss_history.append(compute_wcss(X, labels, centroids))

        # Update step
        new_centroids = update_centroids(X, labels, k, centroids)

        # Convergence check
        shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
        centroids = new_centroids

        if shift < tol:
            break

    return labels, centroids, wcss_history, iteration + 1

### Quick sanity check — single run with random initialisation

In [ ]:
labels_rand, centroids_rand, wcss_hist_rand, n_iters_rand = lloyd_kmeans(
    X_blobs, k=N_CENTERS, init="random", seed=SEED
)

print(f"Converged after {n_iters_rand} iterations")
print(f"Final WCSS: {wcss_hist_rand[-1]:.2f}")
print(f"Cluster sizes: {dict(zip(*np.unique(labels_rand, return_counts=True)))}")

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(wcss_hist_rand, marker="o", markersize=3, color="#4472C4")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("WCSS (inertia)")
axes[0].set_title("K-Means Convergence — WCSS per iteration")

scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_rand,
                          cmap="tab10", s=15, alpha=0.7)
axes[1].scatter(centroids_rand[:, 0], centroids_rand[:, 1],
                c="red", marker="X", s=200, zorder=5, label="Centroids")
axes[1].set_title("K-Means Result (random init)")
axes[1].legend()
plt.tight_layout()
plt.show()

---

## Part 2 — K-Means++ Initialisation & Complete Model

### Why K-Means++ ?

Random initialisation can produce centroids that are very close together, leading to slow convergence and poor local optima. **K-Means++** spreads the initial centroids out by selecting each successive centroid with probability proportional to its **squared distance from the nearest already-chosen centroid**:

$$P(\text{choose } \mathbf{x}_i) \propto \min_j \|\mathbf{x}_i - \boldsymbol{\mu}_j\|^2$$

This ensures centroids start in diverse regions of the feature space, significantly improving the expected final WCSS while adding only $O(nkd)$ overhead.

In [ ]:
def init_kmeans_plus_plus(
    X: np.ndarray,
    k: int,
    rng: np.random.RandomState,
) -> np.ndarray:
    """Initialise k centroids using the K-Means++ algorithm.

    Each successive centroid is chosen with probability proportional to
    its squared distance from the nearest already-selected centroid,
    spreading initial seeds across the data space.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k: Number of clusters.
        rng: NumPy RandomState for reproducibility.

    Returns:
        Initial centroid matrix of shape (k, n_features).
    """
    n = len(X)
    # Step 1: choose first centroid uniformly at random
    first_idx = rng.randint(0, n)
    centroids = [X[first_idx].copy()]

    for _ in range(1, k):
        # Step 2: compute squared distance to nearest chosen centroid
        sq_dists = np.min(
            np.stack([np.sum((X - c) ** 2, axis=1) for c in centroids], axis=1),
            axis=1,
        )
        # Step 3: sample next centroid proportional to squared distance
        probs = sq_dists / sq_dists.sum()
        next_idx = rng.choice(n, p=probs)
        centroids.append(X[next_idx].copy())

    return np.array(centroids)

### Assembling the Full KMeans Class

The complete model wraps multiple restarts (`n_init`) and keeps the best run (lowest WCSS). This is the same strategy used by sklearn.

In [ ]:
class KMeans:
    """K-Means clustering with Lloyd's algorithm and K-Means++ initialisation.

    Runs multiple random restarts and keeps the solution with lowest WCSS.

    Attributes:
        k: Number of clusters.
        init: Initialisation strategy ("random" or "kmeans++").
        max_iter: Maximum Lloyd iterations per restart.
        tol: Convergence tolerance on centroid shift.
        n_init: Number of random restarts.
        seed: Base random seed.
        labels_: Final cluster assignments, shape (n_samples,).
        cluster_centers_: Final centroids, shape (k, n_features).
        inertia_: Best WCSS achieved.
        n_iter_: Iterations used in the best run.
        wcss_history_: WCSS per iteration for the best run.
    """

    def __init__(
        self,
        k: int = 8,
        init: str = "kmeans++",
        max_iter: int = MAX_ITER,
        tol: float = TOL,
        n_init: int = N_INIT,
        seed: int = SEED,
    ) -> None:
        """Initialise K-Means.

        Args:
            k: Number of clusters.
            init: Initialisation strategy.
            max_iter: Maximum iterations per restart.
            tol: Convergence tolerance.
            n_init: Number of restarts.
            seed: Base random seed.
        """
        self.k         = k
        self.init      = init
        self.max_iter  = max_iter
        self.tol       = tol
        self.n_init    = n_init
        self.seed      = seed

    def fit(self, X: np.ndarray) -> "KMeans":
        """Fit K-Means on the data using multiple restarts.

        Args:
            X: Data matrix of shape (n_samples, n_features).

        Returns:
            self
        """
        best_wcss      = float("inf")
        best_labels    = None
        best_centroids = None
        best_history   = None
        best_n_iters   = 0

        for restart in range(self.n_init):
            seed_r = self.seed + restart * 7
            labels, centroids, history, n_iters = lloyd_kmeans(
                X, self.k,
                init=self.init,
                max_iter=self.max_iter,
                tol=self.tol,
                seed=seed_r,
            )
            wcss = history[-1]
            if wcss < best_wcss:
                best_wcss      = wcss
                best_labels    = labels
                best_centroids = centroids
                best_history   = history
                best_n_iters   = n_iters

        self.labels_          = best_labels
        self.cluster_centers_ = best_centroids
        self.inertia_         = best_wcss
        self.n_iter_          = best_n_iters
        self.wcss_history_    = best_history
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Assign new data points to the nearest fitted centroid.

        Args:
            X: Data matrix of shape (n_samples, n_features).

        Returns:
            Cluster labels of shape (n_samples,).
        """
        return assign_clusters(X, self.cluster_centers_)

    def transform(self, X: np.ndarray) -> np.ndarray:
        """Transform data to cluster-distance space.

        Args:
            X: Data matrix of shape (n_samples, n_features).

        Returns:
            Distance matrix of shape (n_samples, k).
        """
        return np.sqrt(np.maximum(0, pairwise_sq_distances(X, self.cluster_centers_)))

### Sanity check — compare random vs K-Means++ initialisation

In [ ]:
results_init = {}
for init_method in ["random", "kmeans++"]:
    km = KMeans(k=N_CENTERS, init=init_method, n_init=N_INIT, seed=SEED)
    t0 = time.perf_counter()
    km.fit(X_blobs)
    elapsed = time.perf_counter() - t0
    results_init[init_method] = {
        "wcss": km.inertia_, "iters": km.n_iter_, "time": elapsed
    }
    print(f"  {init_method:<12} | WCSS={km.inertia_:.2f} | "
          f"iters={km.n_iter_} | time={elapsed:.3f}s")

print("\nK-Means++ usually achieves lower WCSS in fewer iterations.")

---

## Part 3 — Cluster Quality Metrics

### 3.1 Elbow Method

The elbow method plots WCSS against $k$. As $k$ increases, WCSS always decreases — but the rate of decrease slows. The "elbow" is the point where adding another cluster brings diminishing returns.

**Limitation:** The elbow is often ambiguous in real data. Use it alongside silhouette scores.

In [ ]:
def elbow_method(
    X: np.ndarray,
    k_max: int = K_MAX,
    n_init: int = N_INIT,
    seed: int = SEED,
) -> tuple:
    """Compute WCSS for k = 1 to k_max to support elbow-method selection.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k_max: Maximum number of clusters to evaluate.
        n_init: Number of K-Means restarts per k.
        seed: Random seed.

    Returns:
        Tuple of (k_values list, wcss_values list).
    """
    k_values  = list(range(1, k_max + 1))
    wcss_list = []
    for k in k_values:
        km = KMeans(k=k, init="kmeans++", n_init=n_init, seed=seed)
        km.fit(X)
        wcss_list.append(km.inertia_)
        print(f"  k={k:2d}  WCSS={km.inertia_:.2f}")
    return k_values, wcss_list


print("Elbow method — WCSS vs k:")
k_values, wcss_values = elbow_method(X_blobs, k_max=K_MAX)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, wcss_values, marker="o", linewidth=2, color="#4472C4")
ax.axvline(x=N_CENTERS, color="red", linestyle="--", label=f"True k={N_CENTERS}")
ax.set_xlabel("Number of clusters k")
ax.set_ylabel("WCSS (inertia)")
ax.set_title("Elbow Method — WCSS vs k")
ax.legend()
plt.tight_layout()
plt.show()

### 3.2 Silhouette Score

The silhouette score provides a more principled measure of cluster quality. For each point $\mathbf{x}_i$:

- $a_i$ = mean distance to other points **in its own cluster**
- $b_i$ = mean distance to points in the **nearest other cluster**

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)} \in [-1, 1]$$

- $s_i \approx 1$: point is well inside its cluster
- $s_i \approx 0$: point is near a cluster boundary
- $s_i < 0$: point may be in the wrong cluster

The **mean silhouette score** across all points is used for model selection.

In [ ]:
def silhouette_score_scratch(
    X: np.ndarray,
    labels: np.ndarray,
) -> tuple:
    """Compute per-sample and mean silhouette scores from scratch.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        labels: Cluster assignments, shape (n_samples,).

    Returns:
        Tuple of (per_sample_scores array shape (n_samples,), mean_score float).
    """
    n = len(X)
    unique_labels = np.unique(labels)
    k = len(unique_labels)

    if k < 2:
        return np.zeros(n), 0.0

    # Pairwise Euclidean distances — shape (n, n)
    # scipy cdist is allowed in Module 03
    dist_matrix = cdist(X, X, metric="euclidean")

    scores = np.zeros(n)
    for i in range(n):
        own_label = labels[i]
        own_mask  = (labels == own_label)
        own_mask[i] = False   # exclude self

        if own_mask.sum() == 0:
            a_i = 0.0
        else:
            a_i = dist_matrix[i, own_mask].mean()

        # Nearest other cluster mean distance
        b_i = float("inf")
        for label in unique_labels:
            if label == own_label:
                continue
            other_mask = labels == label
            mean_dist  = dist_matrix[i, other_mask].mean()
            if mean_dist < b_i:
                b_i = mean_dist

        denom = max(a_i, b_i)
        scores[i] = (b_i - a_i) / denom if denom > 0 else 0.0

    return scores, float(scores.mean())


def silhouette_sweep(
    X: np.ndarray,
    k_values: list,
    n_init: int = N_INIT,
    seed: int = SEED,
) -> tuple:
    """Compute mean silhouette score for multiple k values.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k_values: List of k values to evaluate.
        n_init: K-Means restarts per k.
        seed: Random seed.

    Returns:
        Tuple of (k_values, silhouette_scores list).
    """
    sil_scores = []
    for k in k_values:
        km = KMeans(k=k, init="kmeans++", n_init=n_init, seed=seed)
        km.fit(X)
        _, mean_sil = silhouette_score_scratch(X, km.labels_)
        # Compare to sklearn for validation
        sk_sil = sklearn_silhouette_score(X, km.labels_)
        print(f"  k={k:2d}  sil_scratch={mean_sil:.4f}  sil_sklearn={sk_sil:.4f}  "
              f"diff={abs(mean_sil - sk_sil):.5f}")
        sil_scores.append(mean_sil)
    return k_values, sil_scores


print("Silhouette scores (scratch vs sklearn validation):")
k_sweep = list(range(2, K_MAX + 1))
k_sil, sil_vals = silhouette_sweep(X_blobs, k_sweep)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_sil, sil_vals, marker="o", linewidth=2, color="#70AD47")
ax.axvline(x=N_CENTERS, color="red", linestyle="--", label=f"True k={N_CENTERS}")
ax.set_xlabel("Number of clusters k")
ax.set_ylabel("Mean Silhouette Score")
ax.set_title("Silhouette Score vs k")
ax.legend()
plt.tight_layout()
plt.show()

### 3.3 Silhouette Plot (per-sample analysis)

A silhouette plot shows per-sample scores within each cluster, revealing which clusters are tight and well-separated vs which are ambiguous.

In [ ]:
# Fit best-k model and produce silhouette plot
km_best = KMeans(k=N_CENTERS, init="kmeans++", n_init=N_INIT, seed=SEED)
km_best.fit(X_blobs)
per_sample_sil, mean_sil = silhouette_score_scratch(X_blobs, km_best.labels_)

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
cluster_colors = plt.cm.tab10(np.linspace(0, 0.5, N_CENTERS))

for j in range(N_CENTERS):
    cluster_sil = per_sample_sil[km_best.labels_ == j]
    cluster_sil.sort()
    size = len(cluster_sil)
    y_upper = y_lower + size
    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, cluster_sil,
        alpha=0.7, color=cluster_colors[j],
        label=f"Cluster {j}",
    )
    ax.text(-0.05, y_lower + size // 2, str(j), fontsize=9)
    y_lower = y_upper + 10

ax.axvline(x=mean_sil, color="red", linestyle="--",
           label=f"Mean={mean_sil:.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Sample (sorted within cluster)")
ax.set_title(f"Silhouette Plot — k={N_CENTERS}")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Mean silhouette score: {mean_sil:.4f}")

---

## Part 4 — Mini-Batch K-Means & Limitations

### 4.1 Mini-Batch K-Means

For large datasets, computing distances from all $n$ points to all $k$ centroids each iteration is expensive. **Mini-Batch K-Means** uses a random mini-batch $B \subset X$ per iteration:

1. Draw a mini-batch of size $|B|$ uniformly at random.
2. **Assignment step:** Assign each batch point to the nearest centroid.
3. **Update step:** Adjust each centroid using a running average weighted by the count of points assigned to it:

$$\boldsymbol{\mu}_j \leftarrow \boldsymbol{\mu}_j + \frac{1}{n_j} \sum_{\mathbf{x}_i \in B, z_i = j} (\mathbf{x}_i - \boldsymbol{\mu}_j)$$

This reduces per-iteration cost to $O(|B| \cdot k \cdot d)$, trading a small accuracy loss for a large speedup.

In [ ]:
class MiniBatchKMeans:
    """Mini-batch variant of K-Means for scalable clustering.

    Uses random mini-batches each iteration instead of the full dataset,
    achieving significant speedup at the cost of a small WCSS increase.

    Attributes:
        k: Number of clusters.
        batch_size: Mini-batch size per iteration.
        max_iter: Maximum number of mini-batch iterations.
        init: Initialisation strategy ("random" or "kmeans++").
        seed: Random seed.
        cluster_centers_: Fitted centroid matrix, shape (k, n_features).
        labels_: Cluster assignments for training data, shape (n_samples,).
        inertia_: Final WCSS on full training data.
    """

    def __init__(
        self,
        k: int = 8,
        batch_size: int = BATCH_SIZE,
        max_iter: int = MAX_ITER,
        init: str = "kmeans++",
        seed: int = SEED,
    ) -> None:
        """Initialise Mini-Batch K-Means.

        Args:
            k: Number of clusters.
            batch_size: Number of samples per mini-batch.
            max_iter: Maximum iterations.
            init: Initialisation strategy.
            seed: Random seed.
        """
        self.k          = k
        self.batch_size = batch_size
        self.max_iter   = max_iter
        self.init       = init
        self.seed       = seed

    def fit(self, X: np.ndarray) -> "MiniBatchKMeans":
        """Fit Mini-Batch K-Means.

        Args:
            X: Data matrix of shape (n_samples, n_features).

        Returns:
            self
        """
        rng = np.random.RandomState(self.seed)
        n, d = X.shape

        # Initialise centroids
        if self.init == "kmeans++":
            centroids = init_kmeans_plus_plus(X, self.k, rng)
        else:
            centroids = init_random(X, self.k, rng)

        # Per-centroid assignment counts (for running average)
        centroid_counts = np.ones(self.k, dtype=float)

        for iteration in range(self.max_iter):
            # Sample a mini-batch
            batch_idx = rng.choice(n, size=min(self.batch_size, n), replace=False)
            X_batch   = X[batch_idx]

            # Assignment step on mini-batch
            batch_labels = assign_clusters(X_batch, centroids)

            # Update step: running-average update
            for j in range(self.k):
                mask = batch_labels == j
                if mask.sum() == 0:
                    continue
                centroid_counts[j] += mask.sum()
                eta = mask.sum() / centroid_counts[j]   # learning rate
                centroids[j] += eta * (X_batch[mask].mean(axis=0) - centroids[j])

        self.cluster_centers_ = centroids
        self.labels_          = assign_clusters(X, centroids)
        self.inertia_         = compute_wcss(X, self.labels_, centroids)
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Assign new data points to the nearest fitted centroid.

        Args:
            X: Data matrix of shape (n_samples, n_features).

        Returns:
            Cluster labels of shape (n_samples,).
        """
        return assign_clusters(X, self.cluster_centers_)

In [ ]:
# Compare Lloyd vs Mini-Batch speed and accuracy
print("Lloyd vs Mini-Batch K-Means comparison:")
print("=" * 55)

# Scale up to 10000 points for a meaningful speed comparison
X_large, _ = make_blobs(n_samples=10_000, centers=N_CENTERS,
                         cluster_std=0.8, random_state=SEED)

results_mb = {}
for name, clf in [
    ("Lloyd (n_init=10)",  KMeans(k=N_CENTERS, n_init=10, seed=SEED)),
    ("Mini-Batch (B=256)", MiniBatchKMeans(k=N_CENTERS, batch_size=256, seed=SEED)),
    ("Mini-Batch (B=64)",  MiniBatchKMeans(k=N_CENTERS, batch_size=64,  seed=SEED)),
]:
    t0 = time.perf_counter()
    clf.fit(X_large)
    elapsed = time.perf_counter() - t0
    results_mb[name] = {"wcss": clf.inertia_, "time": elapsed}
    print(f"  {name:<28} | WCSS={clf.inertia_:,.1f} | Time={elapsed:.3f}s")

print("\nMini-Batch trades a slight WCSS increase for much lower training time.")

### 4.2 Limitations of K-Means

K-Means assumes clusters are **convex, roughly spherical, and equally sized**. It fails on non-convex data (rings, crescents) and data with very different cluster densities. We demonstrate these failure modes.

In [ ]:
# Generate challenging datasets
X_moons, y_moons   = make_moons(n_samples=400, noise=0.08, random_state=SEED)
X_circles, y_circ  = make_circles(n_samples=400, noise=0.05,
                                   factor=0.5, random_state=SEED)
X_aniso = np.dot(
    make_blobs(n_samples=400, centers=3, random_state=SEED)[0],
    [[0.6, -0.6], [-0.4, 0.8]],
)

datasets = [
    ("Two Moons (non-convex)",  X_moons,   y_moons,  2),
    ("Concentric Circles",      X_circles,  y_circ,   2),
    ("Anisotropic Blobs",       X_aniso,   None,      3),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for col, (title, X_ds, y_ds, k_ds) in enumerate(datasets):
    km_fail = KMeans(k=k_ds, init="kmeans++", n_init=10, seed=SEED)
    km_fail.fit(X_ds)

    # Ground truth
    if y_ds is not None:
        axes[0, col].scatter(X_ds[:, 0], X_ds[:, 1], c=y_ds, cmap="tab10",
                              s=15, alpha=0.7)
    else:
        axes[0, col].scatter(X_ds[:, 0], X_ds[:, 1], c="#888888", s=15, alpha=0.7)
    axes[0, col].set_title(f"{title}\n(Ground truth)")

    axes[1, col].scatter(X_ds[:, 0], X_ds[:, 1], c=km_fail.labels_,
                          cmap="tab10", s=15, alpha=0.7)
    axes[1, col].scatter(km_fail.cluster_centers_[:, 0],
                          km_fail.cluster_centers_[:, 1],
                          c="red", marker="X", s=150, zorder=5)
    axes[1, col].set_title(f"K-Means result (k={k_ds})\nWCSS={km_fail.inertia_:.1f}")

plt.suptitle("K-Means Failure Modes — Non-spherical & Anisotropic Data", fontsize=12)
plt.tight_layout()
plt.show()
print("K-Means cannot handle non-convex clusters — see 03-02 (DBSCAN) for a fix.")

### 4.3 Sensitivity to Initialisation

We visualise how different random initialisations lead to different local optima on a harder dataset with overlapping clusters.

In [ ]:
def visualise_init_sensitivity(
    X: np.ndarray,
    k: int,
    n_restarts: int = 6,
    seed: int = SEED,
) -> None:
    """Show different K-Means solutions from different random seeds.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k: Number of clusters.
        n_restarts: Number of different initialisations to show.
        seed: Base random seed.
    """
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, restart in zip(axes.flat, range(n_restarts)):
        labels, centroids, history, n_iters = lloyd_kmeans(
            X, k=k, init="random", max_iter=MAX_ITER, seed=seed + restart * 13
        )
        wcss = history[-1]
        ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="tab10", s=12, alpha=0.6)
        ax.scatter(centroids[:, 0], centroids[:, 1],
                   c="red", marker="X", s=150, zorder=5)
        ax.set_title(f"Seed {seed + restart * 13}  WCSS={wcss:.1f}  iters={n_iters}")
        ax.axis("off")
    plt.suptitle(
        "Sensitivity to Initialisation — Different seeds, same data, same k",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()


# Overlapping blobs — harder for initialisation
X_hard, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.8, random_state=SEED)
print("Visualising initialisation sensitivity on overlapping blobs:")
visualise_init_sensitivity(X_hard, k=4, n_restarts=6)

### 4.4 sklearn Validation

We compare our implementation against scikit-learn's `KMeans` to verify correctness.

In [ ]:
# sklearn validation
sk_km = SklearnKMeans(n_clusters=N_CENTERS, init="k-means++",
                       n_init=N_INIT, random_state=SEED)
sk_km.fit(X_blobs)

km_best.fit(X_blobs)

print("Validation against scikit-learn KMeans:")
print(f"  From-scratch inertia : {km_best.inertia_:.4f}")
print(f"  sklearn      inertia : {sk_km.inertia_:.4f}")
print(f"  Difference           : {abs(km_best.inertia_ - sk_km.inertia_):.4f}")
print()

# Silhouette comparison
scratch_sil = silhouette_score_scratch(X_blobs, km_best.labels_)[1]
sklearn_sil = sklearn_silhouette_score(X_blobs, sk_km.labels_)
print(f"  From-scratch silhouette: {scratch_sil:.4f}")
print(f"  sklearn      silhouette: {sklearn_sil:.4f}")
print()
print("Small differences are expected due to tie-breaking and floating-point order.")

### 4.5 Step-by-Step Convergence Visualisation

We trace the Lloyd iteration frame-by-frame (with selected snapshots) to build intuition for *how* the centroids move during the alternating assignment/update cycle.

In [ ]:
def trace_kmeans_steps(
    X: np.ndarray,
    k: int,
    n_steps: int = 8,
    seed: int = SEED,
) -> list:
    """Run Lloyd's K-Means and record (labels, centroids) at each iteration.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k: Number of clusters.
        n_steps: Maximum number of steps to record.
        seed: Random seed for initialisation.

    Returns:
        List of (labels_array, centroids_array) tuples, one per iteration.
    """
    rng = np.random.RandomState(seed)
    centroids = init_kmeans_plus_plus(X, k, rng)
    history   = []

    for _ in range(n_steps):
        labels         = assign_clusters(X, centroids)
        history.append((labels.copy(), centroids.copy()))
        new_centroids  = update_centroids(X, labels, k, centroids)
        shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
        centroids = new_centroids
        if shift < TOL:
            labels = assign_clusters(X, centroids)
            history.append((labels.copy(), centroids.copy()))
            break

    return history


# Trace on a small dataset for clarity
X_trace, _ = make_blobs(n_samples=200, centers=3, cluster_std=0.9, random_state=SEED)
trace = trace_kmeans_steps(X_trace, k=3, n_steps=6, seed=SEED)

n_frames = min(6, len(trace))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
cmap_trace = plt.cm.tab10

for ax, (labels_t, cents_t) in zip(axes.flat, trace[:n_frames]):
    ax.scatter(X_trace[:, 0], X_trace[:, 1], c=labels_t, cmap="tab10",
               s=18, alpha=0.7)
    ax.scatter(cents_t[:, 0], cents_t[:, 1], c="red", marker="X",
               s=180, zorder=6, label="Centroids")
    wcss_t = compute_wcss(X_trace, labels_t, cents_t)
    ax.set_title(f"Iteration  WCSS={wcss_t:.1f}", fontsize=9)
    ax.axis("off")

plt.suptitle("Lloyd K-Means — Step-by-step convergence", fontsize=12)
plt.tight_layout()
plt.show()
print(f"Converged in {len(trace)} iterations.")

### 4.6 Restart Distribution — WCSS Across Multiple Initialisations

We run 30 random-init restarts and plot the distribution of final WCSS values. This shows why `n_init > 1` is essential and quantifies the benefit of K-Means++ over random initialisation.

In [ ]:
def restart_wcss_distribution(
    X: np.ndarray,
    k: int,
    n_restarts: int = 30,
    seed: int = SEED,
) -> dict:
    """Run many K-Means restarts and collect the WCSS for each.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k: Number of clusters.
        n_restarts: Total number of restarts per initialisation strategy.
        seed: Base random seed.

    Returns:
        Dict mapping init strategy name to list of WCSS values.
    """
    results: dict = {"random": [], "kmeans++": []}
    for restart in range(n_restarts):
        for init_name in ["random", "kmeans++"]:
            _, _, history, _ = lloyd_kmeans(
                X, k=k, init=init_name, seed=seed + restart * 31
            )
            results[init_name].append(history[-1])
    return results


print(f"Running {30} restarts each for random and kmeans++ init...")
restart_results = restart_wcss_distribution(X_blobs, k=N_CENTERS, n_restarts=30)

for init_name, wcss_list in restart_results.items():
    arr = np.array(wcss_list)
    print(f"  {init_name:<12}: mean={arr.mean():.2f}  std={arr.std():.2f}  "
          f"min={arr.min():.2f}  max={arr.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (init_name, wcss_list) in zip(axes, restart_results.items()):
    ax.hist(wcss_list, bins=15, color="#4472C4" if init_name == "random" else "#70AD47",
            edgecolor="black", alpha=0.8)
    ax.axvline(np.min(wcss_list), color="red", linestyle="--",
               label=f"Min={np.min(wcss_list):.1f}")
    ax.set_title(f"WCSS distribution — {init_name} init  (30 restarts)")
    ax.set_xlabel("WCSS")
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.show()
print("K-Means++ concentrates WCSS near the optimum; random init spreads widely.")

### 4.7 Application — Image Colour Quantisation

K-Means can compress an image by replacing the full RGB palette with $k$ representative colours (centroids). Each pixel is remapped to its nearest centroid colour. This is a concrete, visual demonstration of vector quantisation — a concept that reappears in VQ-VAEs (Module 11).

In [ ]:
def generate_synthetic_image(
    height: int = 64,
    width: int = 64,
    n_regions: int = 4,
    seed: int = SEED,
) -> np.ndarray:
    """Generate a synthetic RGB image with distinct colour regions.

    Args:
        height: Image height in pixels.
        width: Image width in pixels.
        n_regions: Number of distinct colour regions.
        seed: Random seed.

    Returns:
        Image array of shape (height, width, 3), dtype float32, range [0, 1].
    """
    rng = np.random.RandomState(seed)
    palette = rng.rand(n_regions, 3).astype(np.float32)
    img = np.zeros((height, width, 3), dtype=np.float32)

    # Assign each pixel a region based on its 2-D position
    cy, cx = height // 2, width // 2
    for r in range(height):
        for c in range(width):
            quad = (r >= cy) * 2 + (c >= cx)
            img[r, c] = palette[quad % n_regions]

    # Add moderate Gaussian noise to make it more interesting
    img += rng.randn(height, width, 3).astype(np.float32) * 0.08
    return np.clip(img, 0, 1)


def colour_quantise(
    img: np.ndarray,
    k_colours: int,
    seed: int = SEED,
) -> tuple:
    """Quantise image colours using K-Means.

    Args:
        img: Input image array of shape (H, W, 3), range [0, 1].
        k_colours: Number of colours in the compressed palette.
        seed: Random seed.

    Returns:
        Tuple of (compressed_image array same shape as img,
                  palette array of shape (k_colours, 3)).
    """
    h, w, c = img.shape
    pixels = img.reshape(-1, c)    # flatten to (H*W, 3)
    km = KMeans(k=k_colours, init="kmeans++", n_init=3, seed=seed)
    km.fit(pixels)
    # Replace each pixel with its centroid colour
    compressed_pixels = km.cluster_centers_[km.labels_]
    compressed_img    = compressed_pixels.reshape(h, w, c)
    return compressed_img.astype(np.float32), km.cluster_centers_


# Generate and quantise
img_orig = generate_synthetic_image(height=64, width=64, n_regions=4, seed=SEED)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes[0, 0].imshow(img_orig)
axes[0, 0].set_title("Original (256 colours)")
axes[0, 0].axis("off")
axes[1, 0].axis("off")

for col_idx, k_col in enumerate([2, 4, 8], start=1):
    compressed, palette = colour_quantise(img_orig, k_colours=k_col)
    mse = float(np.mean((img_orig - compressed) ** 2))
    axes[0, col_idx].imshow(compressed)
    axes[0, col_idx].set_title(f"k={k_col} colours\nMSE={mse:.4f}")
    axes[0, col_idx].axis("off")
    # Show palette
    palette_img = np.repeat(palette[np.newaxis, :, :], 10, axis=0)
    axes[1, col_idx].imshow(palette_img)
    axes[1, col_idx].set_title(f"Palette ({k_col} colours)")
    axes[1, col_idx].axis("off")

plt.suptitle("K-Means Colour Quantisation — Image Compression", fontsize=12)
plt.tight_layout()
plt.show()
print("More centroids = better quality but higher storage cost.")
print("This is the same idea as Vector Quantisation in VQ-VAEs (Module 11).")

### 4.8 K-Means on Higher-Dimensional Data

Most real datasets are not 2-D. We test our implementation on higher-dimensional make_blobs to confirm it scales correctly and that the elbow method still works.

In [ ]:
def kmeans_scaling_experiment(
    dims: list,
    n_samples: int = 500,
    k: int = 4,
    seed: int = SEED,
) -> pd.DataFrame:
    """Measure K-Means WCSS and runtime across increasing dimensionalities.

    Args:
        dims: List of feature dimensions to test.
        n_samples: Samples per dataset.
        k: Number of clusters.
        seed: Random seed.

    Returns:
        DataFrame with columns [Dim, WCSS, Time (s), Silhouette].
    """
    rows = []
    for d in dims:
        X_hd, _ = make_blobs(n_samples=n_samples, centers=k,
                              n_features=d, cluster_std=1.0, random_state=seed)
        scaler_hd = StandardScaler()
        X_hd_sc = scaler_hd.fit_transform(X_hd)
        km_hd = KMeans(k=k, init="kmeans++", n_init=5, seed=seed)
        t0 = time.perf_counter()
        km_hd.fit(X_hd_sc)
        elapsed = time.perf_counter() - t0
        sil = silhouette_score_scratch(X_hd_sc, km_hd.labels_)[1]
        rows.append({
            "Dim": d,
            "WCSS": km_hd.inertia_,
            "Time (s)": elapsed,
            "Silhouette": sil,
        })
        print(f"  dim={d:4d}  WCSS={km_hd.inertia_:8.1f}  "
              f"time={elapsed:.3f}s  sil={sil:.4f}")
    return pd.DataFrame(rows)


from sklearn.preprocessing import StandardScaler

dims_to_test = [2, 5, 10, 20, 50, 100]
print("K-Means scaling experiment across dimensions:")
scaling_df = kmeans_scaling_experiment(dims_to_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scaling_df["Dim"], scaling_df["Time (s)"],
             marker="o", color="#4472C4", linewidth=2)
axes[0].set_xlabel("Dimensionality")
axes[0].set_ylabel("Training Time (s)")
axes[0].set_title("Training Time vs Dimensionality")
axes[0].set_xscale("log")

axes[1].plot(scaling_df["Dim"], scaling_df["Silhouette"],
             marker="o", color="#70AD47", linewidth=2)
axes[1].set_xlabel("Dimensionality")
axes[1].set_ylabel("Mean Silhouette Score")
axes[1].set_title("Cluster Quality vs Dimensionality")
axes[1].set_xscale("log")

plt.tight_layout()
plt.show()
print("\nAs dimensionality grows, silhouette scores drop — curse of dimensionality.")
print("See 03-03 (PCA) for dimensionality reduction before clustering.")

### 4.9 Comprehensive Evaluation Summary

We collect all quality metrics into a single summary table for $k = 2$ through $k = 8$, combining WCSS, silhouette score, and training time to support a data-driven choice of $k$.

In [ ]:
def comprehensive_k_evaluation(
    X: np.ndarray,
    k_values: list,
    n_init: int = N_INIT,
    seed: int = SEED,
) -> pd.DataFrame:
    """Evaluate K-Means for multiple k values using several quality metrics.

    Fits K-Means for each k and computes WCSS, silhouette score,
    normalised WCSS improvement, and training time.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k_values: List of k values to evaluate.
        n_init: Number of K-Means restarts per k.
        seed: Random seed.

    Returns:
        DataFrame with columns [k, WCSS, WCSS Drop (%), Silhouette, Time (s)].
    """
    rows = []
    prev_wcss = None
    for k in k_values:
        km = KMeans(k=k, init="kmeans++", n_init=n_init, seed=seed)
        t0 = time.perf_counter()
        km.fit(X)
        elapsed = time.perf_counter() - t0

        sil = silhouette_score_scratch(X, km.labels_)[1] if k >= 2 else float("nan")
        wcss_drop = 0.0 if prev_wcss is None else (prev_wcss - km.inertia_) / prev_wcss * 100
        prev_wcss = km.inertia_

        rows.append({
            "k":            k,
            "WCSS":         km.inertia_,
            "WCSS Drop (%)": wcss_drop,
            "Silhouette":   sil,
            "Time (s)":     elapsed,
        })

    df = pd.DataFrame(rows)
    return df


print("Comprehensive K-Means evaluation (k = 1 to 8):")
eval_df = comprehensive_k_evaluation(X_blobs, k_values=list(range(1, 9)))
print(eval_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Combined plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
k_plot = eval_df["k"].tolist()

axes[0].plot(k_plot, eval_df["WCSS"].tolist(), marker="o",
             color="#4472C4", linewidth=2)
axes[0].axvline(x=N_CENTERS, color="red", linestyle="--", label=f"True k={N_CENTERS}")
axes[0].set_title("WCSS vs k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("WCSS")
axes[0].legend()

k_sil_plot = [r["k"] for _, r in eval_df.iterrows() if r["k"] >= 2]
sil_plot   = [r["Silhouette"] for _, r in eval_df.iterrows() if r["k"] >= 2]
axes[1].plot(k_sil_plot, sil_plot, marker="o", color="#70AD47", linewidth=2)
axes[1].axvline(x=N_CENTERS, color="red", linestyle="--", label=f"True k={N_CENTERS}")
axes[1].set_title("Silhouette Score vs k")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].legend()

drop_k = [r["k"] for _, r in eval_df.iterrows() if r["k"] >= 2]
drop_v = [r["WCSS Drop (%)"] for _, r in eval_df.iterrows() if r["k"] >= 2]
axes[2].bar(drop_k, drop_v, color="#ED7D31", edgecolor="black", linewidth=0.5)
axes[2].axvline(x=N_CENTERS, color="red", linestyle="--", label=f"True k={N_CENTERS}")
axes[2].set_title("WCSS Reduction (%) vs k")
axes[2].set_xlabel("k")
axes[2].set_ylabel("% WCSS drop from k-1")
axes[2].legend()

plt.suptitle("K-Means Evaluation Dashboard", fontsize=12)
plt.tight_layout()
plt.show()
print(f"\nBest k by silhouette: k={k_sil_plot[int(np.argmax(sil_plot))]}")

### 4.10 Effect of Cluster Overlap on Quality Metrics

Cluster separation directly affects both silhouette scores and elbow sharpness. We sweep over cluster standard deviation and plot how quality metrics degrade as clusters become more overlapping.

In [ ]:
def cluster_overlap_experiment(
    std_values: list,
    k: int = 4,
    n_samples: int = 400,
    n_init: int = 5,
    seed: int = SEED,
) -> pd.DataFrame:
    """Measure K-Means quality as cluster standard deviation (overlap) increases.

    Args:
        std_values: List of cluster_std values to test.
        k: Number of clusters.
        n_samples: Number of samples per dataset.
        n_init: K-Means restarts per configuration.
        seed: Random seed.

    Returns:
        DataFrame with columns [cluster_std, WCSS, Silhouette, Acc (ARI)].
    """
    from sklearn.metrics import adjusted_rand_score

    rows = []
    for std in std_values:
        X_ov, y_ov = make_blobs(n_samples=n_samples, centers=k,
                                 cluster_std=std, random_state=seed)
        km_ov = KMeans(k=k, init="kmeans++", n_init=n_init, seed=seed)
        km_ov.fit(X_ov)
        sil = silhouette_score_scratch(X_ov, km_ov.labels_)[1]
        ari = adjusted_rand_score(y_ov, km_ov.labels_)
        rows.append({
            "cluster_std":  std,
            "WCSS":         km_ov.inertia_,
            "Silhouette":   sil,
            "ARI":          ari,
        })
        print(f"  std={std:.1f}  sil={sil:.4f}  ARI={ari:.4f}  WCSS={km_ov.inertia_:.1f}")

    return pd.DataFrame(rows)


std_sweep = [0.3, 0.6, 0.9, 1.2, 1.5, 2.0, 2.5]
print("Cluster overlap experiment (increasing std = more overlap):")
overlap_df = cluster_overlap_experiment(std_sweep, k=N_CENTERS)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(overlap_df["cluster_std"], overlap_df["Silhouette"],
             marker="o", linewidth=2, color="#4472C4")
axes[0].set_xlabel("Cluster std (overlap)")
axes[0].set_ylabel("Mean Silhouette Score")
axes[0].set_title("Silhouette Score vs Cluster Overlap")

axes[1].plot(overlap_df["cluster_std"], overlap_df["ARI"],
             marker="o", linewidth=2, color="#70AD47")
axes[1].set_xlabel("Cluster std (overlap)")
axes[1].set_ylabel("Adjusted Rand Index")
axes[1].set_title("ARI vs Cluster Overlap")

axes[2].plot(overlap_df["cluster_std"], overlap_df["WCSS"],
             marker="o", linewidth=2, color="#ED7D31")
axes[2].set_xlabel("Cluster std (overlap)")
axes[2].set_ylabel("WCSS")
axes[2].set_title("WCSS vs Cluster Overlap")

plt.suptitle("Effect of Cluster Overlap on K-Means Quality Metrics", fontsize=12)
plt.tight_layout()
plt.show()
print("All metrics degrade as clusters become more overlapping (higher std).")

### 4.11 Centroid Trajectory Tracking

We track how each centroid's position changes across Lloyd iterations. Smooth, rapid convergence to stable positions indicates a good initialisation; erratic movement suggests a poor local minimum.

In [ ]:
def track_centroid_trajectories(
    X: np.ndarray,
    k: int,
    init: str = "kmeans++",
    max_iter: int = 20,
    seed: int = SEED,
) -> tuple:
    """Run K-Means and record centroid positions at every iteration.

    Args:
        X: Data matrix of shape (n_samples, 2). Must be 2-D for plotting.
        k: Number of clusters.
        init: Initialisation strategy.
        max_iter: Maximum iterations.
        seed: Random seed.

    Returns:
        Tuple of (centroid_trajectories list of shape (n_iters+1, k, 2),
                  final_labels array).
    """
    rng = np.random.RandomState(seed)
    if init == "kmeans++":
        centroids = init_kmeans_plus_plus(X, k, rng)
    else:
        centroids = init_random(X, k, rng)

    trajectories = [centroids.copy()]

    for _ in range(max_iter):
        labels         = assign_clusters(X, centroids)
        new_centroids  = update_centroids(X, labels, k, centroids)
        shift = np.max(np.linalg.norm(new_centroids - centroids, axis=1))
        centroids = new_centroids
        trajectories.append(centroids.copy())
        if shift < TOL:
            break

    final_labels = assign_clusters(X, centroids)
    return np.array(trajectories), final_labels


X_traj, _ = make_blobs(n_samples=250, centers=3, cluster_std=1.0, random_state=SEED)
traj_pp, final_labels_pp = track_centroid_trajectories(
    X_traj, k=3, init="kmeans++", seed=SEED
)
traj_rnd, final_labels_rnd = track_centroid_trajectories(
    X_traj, k=3, init="random", seed=SEED + 5
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_traj = ["#E74C3C", "#2ECC71", "#3498DB"]

for ax, traj, labels_f, title in [
    (axes[0], traj_pp,  final_labels_pp,  "K-Means++ init"),
    (axes[1], traj_rnd, final_labels_rnd, "Random init"),
]:
    ax.scatter(X_traj[:, 0], X_traj[:, 1], c=labels_f, cmap="tab10",
               s=15, alpha=0.5, zorder=1)
    for j in range(traj.shape[1]):
        xs = traj[:, j, 0]
        ys = traj[:, j, 1]
        ax.plot(xs, ys, "-o", color=colors_traj[j], linewidth=1.5,
                markersize=4, zorder=3, label=f"Centroid {j}")
        # Mark start and end
        ax.scatter(xs[0], ys[0], s=120, marker="^", color=colors_traj[j],
                   edgecolors="black", linewidths=0.7, zorder=5)
        ax.scatter(xs[-1], ys[-1], s=180, marker="X", color="red",
                   edgecolors="black", linewidths=0.7, zorder=5)
    ax.set_title(f"{title}  ({len(traj)-1} iters)")
    ax.legend(fontsize=7, loc="upper right")

plt.suptitle("Centroid Trajectories — Triangle=Start, X=Final", fontsize=11)
plt.tight_layout()
plt.show()
print("K-Means++ starts centroids in better regions, requiring fewer moves.")

### 4.12 Inertia Ratio — Gap Statistic Intuition

The **gap statistic** compares the observed WCSS against the expected WCSS under a uniform reference distribution. A large gap at $k$ means real clusters exist. We implement a simplified version.

In [ ]:
def gap_statistic(
    X: np.ndarray,
    k_values: list,
    n_ref: int = 10,
    n_init: int = 3,
    seed: int = SEED,
) -> tuple:
    """Compute the gap statistic for a range of k values.

    Compares observed log-WCSS against the expected log-WCSS under a uniform
    reference distribution drawn from the bounding box of the data.

    Args:
        X: Data matrix of shape (n_samples, n_features).
        k_values: List of k values to evaluate.
        n_ref: Number of reference datasets to generate.
        n_init: K-Means restarts per k.
        seed: Random seed.

    Returns:
        Tuple of (gaps list, sk list) where gaps[i] = Gap(k_values[i]) and
        sk is the standard deviation used to compute the threshold.
    """
    rng = np.random.RandomState(seed)
    # Data bounding box for reference draws
    x_min, x_max = X.min(axis=0), X.max(axis=0)

    gaps: list = []
    sks:  list = []

    for k in k_values:
        # Observed log-WCSS
        km_obs = KMeans(k=k, init="kmeans++", n_init=n_init, seed=seed)
        km_obs.fit(X)
        log_wcss_obs = np.log(km_obs.inertia_ + 1e-12)

        # Reference log-WCSS (uniform distribution over bounding box)
        ref_log_wcss = []
        for ref_idx in range(n_ref):
            X_ref = rng.uniform(low=x_min, high=x_max, size=X.shape)
            km_ref = KMeans(k=k, init="kmeans++", n_init=2, seed=seed + ref_idx)
            km_ref.fit(X_ref)
            ref_log_wcss.append(np.log(km_ref.inertia_ + 1e-12))

        ref_arr = np.array(ref_log_wcss)
        gap = ref_arr.mean() - log_wcss_obs
        sk  = np.std(ref_log_wcss) * np.sqrt(1 + 1 / n_ref)
        gaps.append(gap)
        sks.append(sk)
        print(f"  k={k}  gap={gap:.4f}  sk={sk:.4f}")

    return gaps, sks


print("Gap statistic (simplified):")
gap_k = list(range(1, 8))
gaps, sks = gap_statistic(X_blobs, k_values=gap_k, n_ref=8)

# Optimal k: first k where gap(k) >= gap(k+1) - sk(k+1)
optimal_k_gap = gap_k[0]
for i in range(len(gap_k) - 1):
    if gaps[i] >= gaps[i + 1] - sks[i + 1]:
        optimal_k_gap = gap_k[i]
        break

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(gap_k, gaps, yerr=sks, marker="o", capsize=4,
            linewidth=2, color="#4472C4", ecolor="gray")
ax.axvline(x=optimal_k_gap, color="red", linestyle="--",
           label=f"Optimal k={optimal_k_gap} (gap statistic)")
ax.set_xlabel("k")
ax.set_ylabel("Gap(k)")
ax.set_title("Gap Statistic vs k")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Gap statistic suggests k = {optimal_k_gap}  (true k = {N_CENTERS})")

---

## Part 5 — Summary & Lessons Learned

### Key Takeaways

- **Lloyd's algorithm** alternates between two steps — assignment (nearest centroid) and update (centroid = cluster mean) — and is guaranteed to converge in a finite number of iterations, though it may reach a local minimum.
- **K-Means++ initialisation** selects starting centroids proportional to squared distance from existing seeds, dramatically reducing the chance of poor local optima versus random initialisation at the cost of $O(nkd)$ extra work.
- **Elbow method** plots WCSS vs $k$ and looks for a "bend"; **silhouette score** gives a more principled measure of cluster cohesion and separation. Neither is infallible — use both together.
- **Mini-Batch K-Means** replaces the full-data update step with random mini-batches, reducing per-iteration cost from $O(nkd)$ to $O(|B|kd)$ with a small accuracy penalty — critical for large-scale data.
- **K-Means fails on non-convex clusters** (rings, crescents), clusters of very different densities, or highly anisotropic shapes. Module 03-02 (DBSCAN, hierarchical) addresses these failure modes.

### What's Next

→ **03-02 (Hierarchical & Density-Based Clustering)** introduces agglomerative hierarchical clustering and DBSCAN, which can discover arbitrarily-shaped clusters where K-Means fails.

### Going Further

- **K-Medoids** replaces the centroid (mean) with the actual data point closest to the mean, making it robust to outliers.
- **Bisecting K-Means** combines hierarchical and K-Means ideas: recursively split the cluster with highest WCSS, producing a hierarchy.
- **Online (streaming) K-Means** extends the mini-batch idea to true data streams where the dataset size is unknown in advance.